# 17. LLM API Advanced Concepts: Prompt Caching & Batch API

**Duration**: ~2.5 hours  
**Theme**: Bulk Product Enrichment Tool  

---

Every production LLM pipeline eventually hits the same two walls: **cost** and **throughput**.

Imagine you run an e-commerce platform with 10,000 products. Each product needs an SEO summary and feature extraction. At \$0.15/1M input tokens (gpt-4o-mini), a naive loop sends the same massive system prompt 10,000 times — paying full price every time. And each call blocks until the previous one finishes.

Today we'll learn two API-level optimizations that fix exactly this:

| Technique | What it solves | Savings |
|---|---|---|
| **Prompt Caching** | Repeated long prefixes (system prompts) across calls | Up to 50% off input tokens + lower latency |
| **Batch API** | Large async workloads that don't need instant responses | 50% off total cost |

We'll start with the **pain** (naive sequential calls), then earn each optimization step by step.

---
## Setup

In [6]:
# !pip install openai python-dotenv pandas
import pandas as pd
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
import textwrap

import truststore
truststore.inject_into_ssl()



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

MODEL = 'gpt-5-nano'




client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

API key loaded successfully.
OpenAI client ready.


### Our Product Dataset

We'll create a realistic set of product descriptions to work with throughout the notebook.

In [7]:
products = [
    {"id": "P001", "name": "Wireless Bluetooth Headphones", "description": "Over-ear wireless headphones with active noise cancellation, 30-hour battery life, and foldable design. Features Bluetooth 5.3 and multipoint connectivity."},
    {"id": "P002", "name": "Ergonomic Office Chair", "description": "Adjustable lumbar support office chair with breathable mesh back, 4D armrests, seat depth adjustment, and 135-degree recline. Supports up to 300 lbs."},
    {"id": "P003", "name": "Stainless Steel Water Bottle", "description": "Double-wall vacuum insulated 1-liter water bottle. Keeps drinks cold 24 hours or hot 12 hours. BPA-free, leak-proof lid with carrying loop."},
    {"id": "P004", "name": "Mechanical Keyboard", "description": "Compact 75% layout mechanical keyboard with hot-swappable switches, RGB backlighting, gasket mount structure, and PBT double-shot keycaps. USB-C and wireless."},
    {"id": "P005", "name": "Portable Power Station", "description": "500Wh portable power station with LiFePO4 battery, 800W pure sine wave AC output, 100W USB-C PD charging, and solar panel input. Weighs 14 lbs."},
    {"id": "P006", "name": "Smart Doorbell Camera", "description": "2K HDR video doorbell with 180-degree field of view, two-way audio, night vision, and local storage via microSD. Works with Alexa and Google Home."},
    {"id": "P007", "name": "Air Purifier", "description": "HEPA 13 air purifier covering 1,200 sq ft. Auto mode with air quality sensor, whisper-quiet 24dB sleep mode, and filter replacement indicator."},
    {"id": "P008", "name": "Camping Hammock", "description": "Ultralight double camping hammock made from 70D ripstop nylon. Holds 500 lbs. Includes tree straps, carabiners, and stuff sack. Packs down to 1 lb."},
    {"id": "P009", "name": "Electric Standing Desk", "description": "Dual-motor electric standing desk with 60x30 inch bamboo top. Height range 25.6-51.2 inches, 4 memory presets, cable management tray, and anti-collision sensor."},
    {"id": "P010", "name": "Robot Vacuum Cleaner", "description": "LiDAR navigation robot vacuum with 5000Pa suction, auto-empty dock, mop attachment, no-go zones via app, and 180-minute runtime on hardwood and carpet."}
]

df = pd.DataFrame(products)
df

,id,name,description
0,P001,Wireless Bluetooth Headphones,Over-ear wireless headphones with active noise...
1,P002,Ergonomic Office Chair,Adjustable lumbar support office chair with br...
2,P003,Stainless Steel Water Bottle,Double-wall vacuum insulated 1-liter water bot...
3,P004,Mechanical Keyboard,Compact 75% layout mechanical keyboard with ho...
4,P005,Portable Power Station,500Wh portable power station with LiFePO4 batt...
5,P006,Smart Doorbell Camera,2K HDR video doorbell with 180-degree field of...
6,P007,Air Purifier,"HEPA 13 air purifier covering 1,200 sq ft. Aut..."
7,P008,Camping Hammock,Ultralight double camping hammock made from 70...
8,P009,Electric Standing Desk,Dual-motor electric standing desk with 60x30 i...
9,P010,Robot Vacuum Cleaner,LiDAR navigation robot vacuum with 5000Pa suct...


---
## Section 1: The Naive Baseline — Feel the Pain (15 min)

Let's start with the simplest possible approach: a `for` loop calling the API one product at a time. We'll use the **Responses API** (`client.responses.create()`) which is OpenAI's latest and recommended API for text generation.

We'll measure:
- **Latency** per call
- **Token usage** (input vs. output)
- Whether any caching is happening

In [8]:
# A short, simple system prompt — well under 1024 tokens
short_prompt = "You are an assistant that summarizes products for SEO and extracts key features. Return JSON with keys: summary (2 sentences) and features (list of 3-5 strings)."

In [9]:
naive_results = []

for product in products[:5]:  # start with 5 products
    start = time.time()
    
    response = client.responses.create(
        model=MODEL,
        instructions=short_prompt,
        input=f"Product: {product['name']}\nDescription: {product['description']}"
    )
    
    elapsed = time.time() - start
    
    usage = response.usage
    cached = usage.input_tokens_details.cached_tokens if usage.input_tokens_details else 0
    
    naive_results.append({
        "product": product["name"],
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cached_tokens": cached,
        "latency_s": round(elapsed, 2),
        "output_preview": response.output_text[:120] + "..."
    })
    
    print(f"✅ {product['name']}: {elapsed:.2f}s | input={usage.input_tokens} | cached={cached} | output={usage.output_tokens}")

print(f"\n⏱️  Total time: {sum(r['latency_s'] for r in naive_results):.2f}s")

✅ Wireless Bluetooth Headphones: 6.53s | input=85 | cached=0 | output=629
✅ Ergonomic Office Chair: 8.31s | input=88 | cached=0 | output=1161
✅ Stainless Steel Water Bottle: 7.44s | input=86 | cached=0 | output=830
✅ Mechanical Keyboard: 14.16s | input=86 | cached=0 | output=1997
✅ Portable Power Station: 6.69s | input=95 | cached=0 | output=921

⏱️  Total time: 43.13s


In [10]:
pd.DataFrame(naive_results)

,product,input_tokens,output_tokens,cached_tokens,latency_s,output_preview
0,Wireless Bluetooth Headphones,85,629,0,6.53,"{\n ""summary"": ""Experience immersive sound wi..."
1,Ergonomic Office Chair,88,1161,0,8.31,"{\n ""summary"": ""The Ergonomic Office Chair co..."
2,Stainless Steel Water Bottle,86,830,0,7.44,"{\n ""summary"": ""Stainless Steel Water Bottle ..."
3,Mechanical Keyboard,86,1997,0,14.16,"{\n ""summary"": ""Experience a compact 75% layo..."
4,Portable Power Station,95,921,0,6.69,"{\n ""summary"": ""The Portable Power Station de..."


### What do we notice?

1. **`cached_tokens` is 0** for every request. Our system prompt is too short (~40 tokens) — prompt caching requires at least **1,024 tokens** in the prompt prefix.
2. **Each call is sequential** — total time ≈ sum of individual latencies.
3. **We're paying full input price** for the same system prompt every single time.

For 5 products this is fine. For 10,000 products? That's a lot of wasted time and money.

Let's fix both problems.

### 🧠 Knowledge Check 1

1. **Prompt caching kicks in when the prompt prefix is at least:**  
   a) 512 tokens  
   b) 1,024 tokens ✅  
   c) 2,048 tokens  

2. **In the Responses API, the system prompt is set via:**  
   a) `messages=[{"role": "system", ...}]`  
   b) `instructions=...` ✅  
   c) `system_prompt=...`  

3. **Cached token counts appear in the response under:**  
   a) `usage.total_tokens`  
   b) `usage.input_tokens_details.cached_tokens` ✅  
   c) `metadata.cache_hits`

---
## Section 2: Prompt Caching Deep Dive (30 min)

### How Prompt Caching Works

Prompt caching is **automatic** — no code changes required. Here's the mechanism:

```
Request 1:  [=== system prompt (1500 tokens) ===][user: Product A]
                     ↓ computed & cached

Request 2:  [=== system prompt (1500 tokens) ===][user: Product B]
                     ↓ cache HIT! skip recomputation

Request 3:  [=== system prompt (1500 tokens) ===][user: Product C]
                     ↓ cache HIT! 
```

**Key rules:**
- Caching starts at **1,024 tokens** and grows in **128-token chunks**
- Only **exact prefix matches** hit the cache — change one character and the cache misses
- Supported on: gpt-4o, gpt-4o-mini, o1, o3, and newer models
- Cached tokens get a **50% discount** on input token pricing
- Caches evict after **5–10 minutes** of inactivity (up to 1 hour in off-peak)
- **No extra cost** to write to the cache — it's free

### Step 1: Build a Long System Prompt (≥1,024 tokens)

To trigger caching, we need a system prompt with at least 1,024 tokens. In production, this is natural — detailed style guides, few-shot examples, output schemas, and domain rules easily exceed this threshold.

In [16]:
long_system_prompt = """
You are an expert e-commerce product content specialist working for a major online marketplace.
Your job is to create SEO-optimized product summaries and extract structured feature data.

=== OUTPUT FORMAT ===
You MUST respond with valid JSON only. No markdown, no explanation, just the JSON object.
{
  "seo_summary": "A compelling 2-3 sentence summary optimized for search engines. Include the product name, primary benefit, and a call-to-action phrase. Use active voice throughout.",
  "features": ["Feature 1 as a concise bullet point", "Feature 2", ...],
  "category": "The most specific product category",
  "target_audience": "Who this product is best suited for",
  "keywords": ["seo", "keyword", "list"]
}

=== WRITING STYLE GUIDE ===
1. Always write in active voice. BAD: "The battery is designed to last 30 hours." GOOD: "Enjoy 30 hours of uninterrupted battery life."
2. Lead with the primary benefit, not the feature. BAD: "Has Bluetooth 5.3" GOOD: "Connect seamlessly to all your devices with Bluetooth 5.3"
3. Use power words: discover, transform, elevate, essential, premium, revolutionary, seamless
4. Avoid superlatives without evidence. Don't say "best" or "greatest" unless the product description provides comparative data.
5. Keep sentences between 15-25 words for readability.
6. Include at least one emotional trigger word per summary (comfort, confidence, peace of mind, freedom, etc.)
7. End the SEO summary with an implicit call-to-action ("Upgrade your...", "Transform your...", "Experience...")

=== FEATURE EXTRACTION RULES ===
1. Extract 3-5 features maximum. Prioritize by consumer impact.
2. Each feature should be a standalone phrase (not a full sentence). 5-12 words each.
3. Always include the primary spec (battery life, weight, capacity, etc.) with units.
4. Group related specs into a single feature when possible.
5. Order features from most important to least important for the target consumer.

=== CATEGORY TAXONOMY ===
Use the most specific category from this hierarchy:
- Electronics > Audio > Headphones > Over-Ear / In-Ear / On-Ear
- Electronics > Computers > Peripherals > Keyboards / Mice / Monitors
- Electronics > Smart Home > Security / Lighting / Climate / Cleaning
- Furniture > Office > Chairs / Desks / Storage
- Outdoor & Sports > Camping > Shelter / Sleep / Cook / Accessories
- Kitchen & Home > Drinkware / Cookware / Appliances
- Power & Energy > Portable Power / Solar / Batteries

=== KEYWORD GUIDELINES ===
1. Generate 5-8 SEO keywords per product.
2. Include both short-tail (1-2 words) and long-tail (3-5 words) keywords.
3. Always include the product type, primary feature, and target use case.
4. Avoid brand names unless provided in the product description.
5. Include common misspellings or alternative terms consumers might search for.

=== FEW-SHOT EXAMPLES ===

EXAMPLE INPUT:
Product: Running Shoes
Description: Lightweight mesh running shoes with responsive foam cushioning, reinforced heel counter, and reflective details. Available in sizes 6-14. Weighs 8.5 oz.

EXAMPLE OUTPUT:
{
  "seo_summary": "Hit the pavement with confidence in these ultralight running shoes featuring responsive foam cushioning that absorbs impact mile after mile. The breathable mesh upper and reinforced heel deliver the perfect balance of comfort and stability. Elevate your running experience today.",
  "features": ["Responsive foam cushioning for impact absorption", "Breathable lightweight mesh upper at 8.5 oz", "Reinforced heel counter for stability", "Reflective details for low-light visibility", "Available in sizes 6 through 14"],
  "category": "Outdoor & Sports > Running > Footwear",
  "target_audience": "Recreational and intermediate runners seeking lightweight, cushioned footwear",
  "keywords": ["running shoes", "lightweight running shoes", "cushioned running shoes", "mesh running shoes", "reflective running shoes", "responsive foam shoes", "8.5 oz running shoes"]
}

EXAMPLE INPUT:
Product: Yoga Mat
Description: 6mm thick non-slip yoga mat made from natural tree rubber with alignment markings. Dimensions 72x26 inches. Includes carrying strap. Free from PVC, latex, and silicone.

EXAMPLE OUTPUT:
{
  "seo_summary": "Transform your practice on this premium natural rubber yoga mat with built-in alignment markings that guide your poses with precision. The 6mm thick non-slip surface provides the perfect cushion-to-grip ratio for any flow style. Discover your best practice with this eco-friendly essential.",
  "features": ["6mm natural tree rubber with non-slip grip", "Built-in alignment markings for guided practice", "72x26 inch full-size dimensions", "Eco-friendly: PVC, latex, and silicone free", "Includes carrying strap for portability"],
  "category": "Outdoor & Sports > Yoga > Mats",
  "target_audience": "Yoga practitioners of all levels seeking an eco-conscious, non-slip mat",
  "keywords": ["yoga mat", "non-slip yoga mat", "natural rubber yoga mat", "alignment yoga mat", "eco-friendly yoga mat", "6mm yoga mat", "yoga mat with markings"]
}

=== IMPORTANT REMINDERS ===
- Output ONLY the JSON object. No preamble, no markdown code fences, no explanation.
- If the product description is vague, make reasonable inferences but do not fabricate specific numbers.
- Use American English spelling throughout.
- Ensure all JSON keys are exactly as shown in the format specification above.
"""

# Rough token count check (1 token ≈ 4 chars for English)
estimated_tokens = len(long_system_prompt) / 4
print(f"Estimated system prompt tokens: ~{int(estimated_tokens)}")
print(f"Meets 1024-token minimum: {'✅ Yes' if estimated_tokens >= 1024 else '❌ No — need to expand'}")

Estimated system prompt tokens: ~1344
Meets 1024-token minimum: ✅ Yes


### Step 2: Observe Caching in Action

Let's send the same long system prompt across multiple requests and watch the `cached_tokens` field.

In [12]:
cache_results = []

for i, product in enumerate(products[:5]):
    start = time.time()
    
    response = client.responses.create(
        model=MODEL,
        instructions=long_system_prompt,
        input=f"Product: {product['name']}\nDescription: {product['description']}"
    )
    
    elapsed = time.time() - start
    usage = response.usage
    cached = usage.input_tokens_details.cached_tokens if usage.input_tokens_details else 0
    
    cache_results.append({
        "request_num": i + 1,
        "product": product["name"],
        "input_tokens": usage.input_tokens,
        "cached_tokens": cached,
        "cache_hit_pct": f"{(cached / usage.input_tokens * 100):.1f}%" if usage.input_tokens > 0 else "0%",
        "output_tokens": usage.output_tokens,
        "latency_s": round(elapsed, 2)
    })
    
    print(f"Request {i+1}: {product['name'][:30]:30s} | latency={elapsed:.2f}s | cached={cached}/{usage.input_tokens} tokens")

print(f"\n⏱️  Total: {sum(r['latency_s'] for r in cache_results):.2f}s")

Request 1: Wireless Bluetooth Headphones  | latency=17.18s | cached=0/1249 tokens
Request 2: Ergonomic Office Chair         | latency=26.99s | cached=1152/1252 tokens
Request 3: Stainless Steel Water Bottle   | latency=34.79s | cached=1152/1250 tokens
Request 4: Mechanical Keyboard            | latency=20.71s | cached=0/1250 tokens
Request 5: Portable Power Station         | latency=24.18s | cached=1152/1259 tokens

⏱️  Total: 123.85s


In [13]:
cache_df = pd.DataFrame(cache_results)
cache_df

,request_num,product,input_tokens,cached_tokens,cache_hit_pct,output_tokens,latency_s
0,1,Wireless Bluetooth Headphones,1249,0,0.0%,2526,17.18
1,2,Ergonomic Office Chair,1252,1152,92.0%,3257,26.99
2,3,Stainless Steel Water Bottle,1250,1152,92.2%,4945,34.79
3,4,Mechanical Keyboard,1250,0,0.0%,3185,20.71
4,5,Portable Power Station,1259,1152,91.5%,3667,24.18


### What to look for:

- **Request 1**: `cached_tokens = 0` — this is the **cold start**. The cache is being populated.
- **Request 2+**: `cached_tokens > 0` — the system prompt prefix is being reused!
- **Latency**: Later requests should be noticeably faster.

> ⚠️ **Note**: Caching is best-effort. OpenAI routes requests to machines based on a hash of the prompt prefix. Sometimes you may not get a cache hit if the request lands on a different machine. Sending requests in rapid succession to the same org with the same prefix maximizes hit rates.

### Step 3: Cache Miss Experiment

What happens if we change even a single word in the system prompt? The cache requires **exact prefix match** — any change invalidates it.

In [17]:
# First, send one more request with the ORIGINAL prompt to warm the cache
response_warm = client.responses.create(
    model=MODEL,
    instructions=long_system_prompt,
    input=f"Product: {products[5]['name']}\nDescription: {products[5]['description']}"
)
cached_warm = response_warm.usage.input_tokens_details.cached_tokens if response_warm.usage.input_tokens_details else 0
print(f"Original prompt  → cached: {cached_warm}/{response_warm.usage.input_tokens}")

# Now modify ONE WORD in the system prompt
modified_prompt = long_system_prompt.replace("expert e-commerce", "skilled e-commerce")  # just one word changed!

response_miss = client.responses.create(
    model=MODEL,
    instructions=modified_prompt,
    input=f"Product: {products[5]['name']}\nDescription: {products[5]['description']}"
)
cached_miss = response_miss.usage.input_tokens_details.cached_tokens if response_miss.usage.input_tokens_details else 0
print(f"Modified prompt   → cached: {cached_miss}/{response_miss.usage.input_tokens}")
print(f"\n💡 Changing one word in the prefix caused a cache {'MISS ❌' if cached_miss < cached_warm else 'HIT ✅'}")

Original prompt  → cached: 1152/1254
Modified prompt   → cached: 1152/1254

💡 Changing one word in the prefix caused a cache HIT ✅


### Takeaway: Prompt Versioning Matters

In production, this means:
- **Freeze your system prompt** — treat it as versioned config, not something you edit casually
- **Put dynamic content at the END** of the input, not embedded in the system prompt
- **Static content first**: instructions → examples → schema → then the variable user input

### Step 4: Visualize the Cost Impact

In [21]:
INPUT_COST_PER_M = 0.05
CACHED_COST_PER_M = 0.005
OUTPUT_COST_PER_M = 0.40


def calculate_cost_from_averages(
    num_requests,
    avg_input_tokens,
    avg_cached_tokens,
    avg_output_tokens,
):
    if avg_cached_tokens > avg_input_tokens:
        raise ValueError("avg_cached_tokens cannot exceed avg_input_tokens")

    total_input = num_requests * avg_input_tokens
    total_cached = num_requests * avg_cached_tokens
    total_uncached = total_input - total_cached
    total_output = num_requests * avg_output_tokens

    input_cost = (
        (total_uncached / 1_000_000) * INPUT_COST_PER_M
        + (total_cached / 1_000_000) * CACHED_COST_PER_M
    )
    output_cost = (total_output / 1_000_000) * OUTPUT_COST_PER_M
    total_cost = input_cost + output_cost

    return {
        "num_requests": num_requests,
        "total_input_tokens": total_input,
        "cached_tokens": total_cached,
        "uncached_tokens": total_uncached,
        "output_tokens": total_output,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


def compare_scenarios(num_requests, naive, cached):
    naive_cost = calculate_cost_from_averages(num_requests, **naive)
    cached_cost = calculate_cost_from_averages(num_requests, **cached)

    savings = naive_cost["total_cost"] - cached_cost["total_cost"]
    savings_pct = (savings / naive_cost["total_cost"]) * 100 if naive_cost["total_cost"] else 0

    return naive_cost, cached_cost, savings, savings_pct


NUM_REQUESTS = 1_000_000

naive_profile = {
    "avg_input_tokens": 4000,
    "avg_cached_tokens": 0,
    "avg_output_tokens": 100,
}

cached_profile = {
    "avg_input_tokens": 4000,
    "avg_cached_tokens": 3500,
    "avg_output_tokens": 100,
}

naive_cost, cached_cost, savings, savings_pct = compare_scenarios(
    NUM_REQUESTS,
    naive_profile,
    cached_profile,
)

print("Naive total cost:  $%.2f" % naive_cost["total_cost"])
print("Cached total cost: $%.2f" % cached_cost["total_cost"])
print("Savings:           $%.2f (%.2f%%)" % (savings, savings_pct))

Naive total cost:  $240.00
Cached total cost: $82.50
Savings:           $157.50 (65.62%)


OpenAI’s docs say prompt caching can reduce latency by up to 80%, because repeated prompt prefixes can be reused instead of recomputed from scratch.

### 🧠 Knowledge Check 2

1. **Cached prompt tokens get reused when:**  
   a) The user message is identical  
   b) The prompt prefix (≥1024 tokens) is unchanged ✅  
   c) The model is GPT-3.5-turbo  

2. **Changing one word in the system prompt will:**  
   a) Still hit the cache  
   b) Cause a cache miss ✅  
   c) Improve latency  

3. **Why is prompt caching especially useful for RAG or summarization pipelines?**  
   a) The same static system prompt gets reused across thousands of queries ✅  
   b) Every query is unique  
   c) Because embeddings are cheaper

---
## Section 3: Batch API Deep Dive (30 min)

Prompt caching makes individual calls cheaper. But we're still calling them **one at a time**.

What if you have 10,000 products to process and don't need results instantly? The **Batch API** lets you:

1. Bundle thousands of requests into a single **JSONL file**
2. Upload and submit them as one **batch job**
3. Poll until done (up to 24 hours, usually much faster)
4. Download all results at once

The reward? **50% cost discount** and **separate rate limits** that won't throttle your live traffic.

```
┌─────────────────┐     ┌──────────────┐     ┌──────────────┐
│  Prepare JSONL   │────▶│  Upload file  │────▶│ Create batch  │
│  (local)         │     │  (Files API)  │     │ (Batches API) │
└─────────────────┘     └──────────────┘     └──────┬───────┘
                                                     │
                                                     ▼
                         ┌──────────────┐     ┌──────────────┐
                         │ Download file │◀────│ Poll status   │
                         │ (Files API)   │     │ until done    │
                         └──────────────┘     └──────────────┘
```

Some real-life cases where Batch API makes sense:

**1. Support-ticket triage at the end of the hour/day.**
A company dumps thousands of Zendesk or email tickets into a batch job to classify intent, urgency, language, likely team, and maybe draft a suggested reply. This fits Batch because it is built for **asynchronous jobs**, offers **50% lower costs**, and is meant for workloads that **do not require immediate responses**. ([OpenAI Developers][1])

**2. E-commerce catalog enrichment.**
Marketplaces use it to generate product titles, bullet summaries, attribute extraction, taxonomy tags, or image captions across large catalogs. OpenAI’s cookbook explicitly calls out **tagging, captioning, or enriching content on a marketplace or blog** as a Batch-style workload. ([OpenAI Developers][2])

**3. Large-scale document summarization or translation.**
If you have a backlog of articles, PDFs, legal docs, or internal reports and want summaries, translations, or structured extraction overnight, Batch is a good fit. The cookbook lists **summaries or translations for collections of documents or articles** as a core example. ([OpenAI Developers][2])

**4. Customer-feedback analytics.**
Teams often run daily or weekly jobs over NPS comments, app-store reviews, survey free text, and call transcripts to do sentiment analysis, topic clustering, complaint extraction, and trend reporting. The cookbook specifically mentions **sentiment analysis on large datasets of customer feedback**. ([OpenAI Developers][2])

**5. Offline moderation and safety sweeps.**
A platform can scan a backlog of posts, comments, images, or uploads for policy violations instead of scoring every item live. Batch is useful when you want higher throughput and lower cost, and you can tolerate the documented **24-hour completion window**. ([OpenAI Developers][1])

**6. Eval pipelines and regression testing.**
Model teams use Batch to run the same prompt set over thousands of test cases after changing prompts, tools, or model versions. This is one of the classic “not user-facing, high-volume” jobs, and Batch supports large JSONL input files with up to **50,000 requests per file**. ([OpenAI Platform][3])

**7. CRM / lead enrichment.**
Sales and ops teams can process lead records in bulk to normalize company descriptions, classify industry, summarize account notes, or extract fields from messy text. This is basically the same pattern as data enrichment workloads that the docs position for Batch and related lower-cost async processing. ([OpenAI Developers][1])

**8. Backfill jobs after launching a new feature.**
Example: you add “AI summaries” to your app and need to generate summaries for the last 2 million old records. Batch is ideal for this kind of one-time backlog processing because it uses a **separate pool of significantly higher rate limits** than normal synchronous traffic. ([OpenAI Developers][1])

Where Batch is usually **not** the right tool: live chat, autocomplete, agent loops, or anything user-facing where waiting minutes or hours is unacceptable. The official guide is explicit that it is for jobs that **don’t require immediate responses** and uses a **24-hour turnaround window**. ([OpenAI Developers][1])

A simple rule: we use Batch when the work is **high-volume, offline, and delay-tolerant**. Use regular synchronous APIs when a human is waiting.

[1]: https://developers.openai.com/api/docs/guides/batch/?utm_source=chatgpt.com "Batch API"
[2]: https://developers.openai.com/cookbook/examples/batch_processing/?utm_source=chatgpt.com "Batch processing with the Batch API"
[3]: https://platform.openai.com/docs/api-reference/batch?_clear=true&lang=node&utm_source=chatgpt.com "Batch | OpenAI API Reference"


**JSON** is one structured value.
**JSONL** is many JSON values, one per line.


1. Example 1


Example JSON:

```json
{
  "users": [
    { "id": 1, "name": "A" },
    { "id": 2, "name": "B" }
  ]
}
```

Example JSONL:

```json
{"id": 1, "name": "A"}
{"id": 2, "name": "B"}
```


2. Exmple 2

One whole document:

```json
{
  "orders": [
    {
      "order_id": "A1001",
      "customer": "Shivam",
      "amount": 499,
      "status": "paid"
    },
    {
      "order_id": "A1002",
      "customer": "Riya",
      "amount": 899,
      "status": "pending"
    },
    {
      "order_id": "A1003",
      "customer": "Arjun",
      "amount": 1299,
      "status": "shipped"
    }
  ]
}
```

---


One record per line:

```json
{"order_id":"A1001","customer":"Shivam","amount":499,"status":"paid"}
{"order_id":"A1002","customer":"Riya","amount":899,"status":"pending"}
{"order_id":"A1003","customer":"Arjun","amount":1299,"status":"shipped"}
```

3. Example 3

```json
[
  {"event":"login","user":"u1"},
  {"event":"search","user":"u2"},
  {"event":"logout","user":"u1"}
]
```

Adding one more record means editing the array structure carefully.

With JSONL, you just append another line:

```json
{"event":"login","user":"u1"}
{"event":"search","user":"u2"}
{"event":"logout","user":"u1"}
{"event":"purchase","user":"u3"}
```



### Step 1: Prepare the JSONL File

Each line in the JSONL file is one API request. The format mirrors what you'd send to the Responses API, wrapped in an envelope with `custom_id`, `method`, `url`, and `body`.

The Batch API supports the `/v1/responses` endpoint, so we can use the same Responses API syntax we've been using.

In [22]:
batch_requests = []

for product in products:
    request = {
        "custom_id": f"product-{product['id']}",
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": MODEL,
            "instructions": long_system_prompt,
            "input": f"Product: {product['name']}\nDescription: {product['description']}"
        }
    }
    batch_requests.append(request)

# Write to JSONL file
batch_file_path = "batch_input.jsonl"
with open(batch_file_path, "w") as f:
    for req in batch_requests:
        f.write(json.dumps(req) + "\n")

print(f"Created {batch_file_path} with {len(batch_requests)} requests")
print(f"File size: {os.path.getsize(batch_file_path):,} bytes")

Created batch_input.jsonl with 10 requests
File size: 59,140 bytes


In [23]:
# Let's peek at what one line looks like
with open(batch_file_path) as f:
    first_line = json.loads(f.readline())

# Show structure (truncate the long prompt for readability)
preview = first_line.copy()
preview["body"] = {**preview["body"], "instructions": preview["body"]["instructions"][:100] + "..."}
print(json.dumps(preview, indent=2))

{
  "custom_id": "product-P001",
  "method": "POST",
  "url": "/v1/responses",
  "body": {
    "model": "gpt-5-nano",
    "instructions": "\nYou are an expert e-commerce product content specialist working for a major online marketplace.\nYou...",
    "input": "Product: Wireless Bluetooth Headphones\nDescription: Over-ear wireless headphones with active noise cancellation, 30-hour battery life, and foldable design. Features Bluetooth 5.3 and multipoint connectivity."
  }
}


### Step 2: Upload the File & Create the Batch Job

In [24]:
# Upload the JSONL file
batch_file = client.files.create(
    file=open(batch_file_path, "rb"),
    purpose="batch"
)
print(f"Uploaded file: {batch_file.id}")
print(f"Status: {batch_file.status}")

Uploaded file: file-8Cu5s8A15zybpRpc6kcn2g
Status: processed


In [25]:
# Create the batch job
batch_job = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
    metadata={
        "description": "Product enrichment demo — 10 products"
    }
)

print(f"Batch ID: {batch_job.id}")
print(f"Status: {batch_job.status}")
print(f"Created at: {batch_job.created_at}")

Batch ID: batch_69b23444b83081909b0a593c4030a0f1
Status: validating
Created at: 1773286468


### Step 3: Poll Until Complete

Batch jobs are asynchronous. We poll the status until it reaches `completed` (or `failed`/`expired`).

Possible statuses: `validating` → `in_progress` → `finalizing` → `completed`

In [27]:
poll_start = time.time()

while True:
    job = client.batches.retrieve(batch_job.id)
    elapsed = time.time() - poll_start
    
    completed = job.request_counts.completed if job.request_counts else 0
    total = job.request_counts.total if job.request_counts else 0
    
    print(f"[{elapsed:6.1f}s] Status: {job.status:15s} | Progress: {completed}/{total}")
    
    if job.status in ["completed", "failed", "expired", "cancelled"]:
        break
    
    time.sleep(10)

print(f"\n✅ Batch finished in {elapsed:.1f}s with status: {job.status}")
if job.request_counts:
    print(f"   Completed: {job.request_counts.completed} | Failed: {job.request_counts.failed}")

[   1.6s] Status: in_progress     | Progress: 0/10
[  13.4s] Status: in_progress     | Progress: 0/10
[  24.9s] Status: in_progress     | Progress: 0/10
[  36.5s] Status: in_progress     | Progress: 0/10
[  48.3s] Status: in_progress     | Progress: 0/10
[  59.9s] Status: in_progress     | Progress: 0/10
[  71.6s] Status: in_progress     | Progress: 0/10
[  83.1s] Status: in_progress     | Progress: 0/10
[  94.7s] Status: in_progress     | Progress: 0/10
[ 106.5s] Status: in_progress     | Progress: 0/10
[ 118.1s] Status: in_progress     | Progress: 0/10
[ 130.1s] Status: in_progress     | Progress: 0/10
[ 141.9s] Status: in_progress     | Progress: 0/10
[ 153.5s] Status: in_progress     | Progress: 0/10
[ 165.2s] Status: in_progress     | Progress: 0/10
[ 176.8s] Status: in_progress     | Progress: 0/10
[ 188.4s] Status: in_progress     | Progress: 0/10
[ 200.0s] Status: in_progress     | Progress: 0/10
[ 211.7s] Status: in_progress     | Progress: 0/10
[ 223.4s] Status: in_progress  

### Step 4: Download and Parse Results

In [28]:
if job.status == "completed" and job.output_file_id:
    # Download the output file
    output_content = client.files.content(job.output_file_id)
    output_text = output_content.text
    
    # Save locally
    with open("batch_output.jsonl", "w") as f:
        f.write(output_text)
    
    # Parse each line
    batch_results = []
    for line in output_text.strip().split("\n"):
        result = json.loads(line)
        batch_results.append(result)
    
    print(f"Downloaded {len(batch_results)} results")
else:
    print(f"Batch did not complete successfully. Status: {job.status}")
    if job.error_file_id:
        error_content = client.files.content(job.error_file_id)
        print("Errors:", error_content.text[:500])

Downloaded 10 results


In [29]:
# Examine one result in detail
if batch_results:
    sample = batch_results[0]
    print(f"Custom ID: {sample['custom_id']}")
    print(f"Status code: {sample['response']['status_code']}")
    
    # The response body contains the same structure as a Responses API call
    body = sample['response']['body']
    
    # Extract the output text from the response
    output_items = body.get('output', [])
    for item in output_items:
        if item.get('type') == 'message':
            for content in item.get('content', []):
                if content.get('type') == 'output_text':
                    print(f"\nOutput text (first 300 chars):\n{content['text'][:300]}...")
    
    # Check usage
    usage = body.get('usage', {})
    print(f"\nUsage: input={usage.get('input_tokens', 'N/A')}, output={usage.get('output_tokens', 'N/A')}")
    print(f"Cached: {usage.get('input_tokens_details', {}).get('cached_tokens', 'N/A')}")

Custom ID: product-P001
Status code: 200

Output text (first 300 chars):
{
  "seo_summary": "Experience immersive sound with Wireless Bluetooth Headphones featuring active noise cancellation and up to 30-hour battery life. Seamlessly connect via Bluetooth 5.3 with multipoint pairing in a foldable, portable design. Upgrade your listening experience today.",
  "features": ...

Usage: input=1249, output=2460
Cached: 1152


In [30]:
# Parse all batch results into a clean DataFrame
parsed_results = []

for result in batch_results:
    custom_id = result["custom_id"]
    status_code = result["response"]["status_code"]
    body = result["response"]["body"]
    
    # Extract output text
    output_text = ""
    for item in body.get("output", []):
        if item.get("type") == "message":
            for content in item.get("content", []):
                if content.get("type") == "output_text":
                    output_text = content["text"]
    
    usage = body.get("usage", {})
    
    parsed_results.append({
        "custom_id": custom_id,
        "status": status_code,
        "input_tokens": usage.get("input_tokens", 0),
        "cached_tokens": usage.get("input_tokens_details", {}).get("cached_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
        "output_preview": output_text[:100]
    })

batch_df = pd.DataFrame(parsed_results)
batch_df

,custom_id,status,input_tokens,cached_tokens,output_tokens,output_preview
0,product-P001,200,1249,1152,2460,"{\n ""seo_summary"": ""Experience immersive soun..."
1,product-P002,200,1252,0,3032,"{\n ""seo_summary"": ""Experience lasting comfor..."
2,product-P003,200,1250,1152,3286,"{\n ""seo_summary"": ""Keep beverages cold for 2..."
3,product-P004,200,1250,1152,3736,"{\n ""seo_summary"": ""Transform your typing wit..."
4,product-P005,200,1259,0,3232,"{\n ""seo_summary"": ""Power wherever you go wit..."
5,product-P006,200,1254,1152,3022,"{\n ""seo_summary"": ""Elevate your home securit..."
6,product-P007,200,1253,0,3205,"{\n ""seo_summary"": ""Transform your home's air..."
7,product-P008,200,1256,0,2736,"{\n ""seo_summary"": ""Experience restful outdoo..."
8,product-P009,200,1259,0,2949,"{\n ""seo_summary"": ""Elevate your workspace wi..."
9,product-P010,200,1252,1152,2938,"{\n ""seo_summary"": ""Experience effortless dee..."


### Batch API: Key Limitations to Know

| Aspect | Detail |
|---|---|
| **Latency** | Up to 24 hours (usually minutes to a few hours) |
| **Streaming** | Not supported — results come as a complete file |
| **Max requests** | 50,000 per batch |
| **Max file size** | 200 MB per input file |
| **Single model** | All requests in one file must use the same model |
| **Rate limits** | Separate from synchronous limits (won't affect your live app) |

### 🧠 Knowledge Check 3

1. **Batch API requests are uploaded as:**  
   a) CSV  
   b) JSONL file ✅  
   c) Pickle  

2. **Batch API jobs return results via:**  
   a) Direct HTTP response  
   b) An output file you download ✅  
   c) Streaming tokens  

3. **The Batch API discount is roughly:**  
   a) 5%  
   b) 25%  
   c) 50% ✅

---
## Section 4: Combining Prompt Caching + Batch API (25 min)

Here's the real power move: **use both together**.

- Long static system prompt → triggers **prompt caching** (50% off cached input tokens)
- Sent via **Batch API** → additional **50% off everything**

The savings stack. Let's build a proper comparison.

### Expanded Product Dataset

Let's scale up to a more realistic batch size.

In [31]:
# Extended product catalog for batch demo
more_products = products + [
    {"id": "P011", "name": "Wireless Earbuds", "description": "True wireless earbuds with ANC, transparency mode, 8-hour battery per charge (32 with case), IPX5 water resistance, and touch controls. Bluetooth 5.3."},
    {"id": "P012", "name": "4K Webcam", "description": "4K Ultra HD webcam with auto-framing, built-in ring light, dual noise-cancelling microphones, and USB-C plug-and-play. Compatible with all major video platforms."},
    {"id": "P013", "name": "Electric Kettle", "description": "Variable temperature electric kettle with 1.7L capacity, 6 preset temps (160°F-212°F), keep-warm function, gooseneck pour spout, and rapid boil in under 5 minutes."},
    {"id": "P014", "name": "Hiking Backpack", "description": "45L hiking backpack with ventilated back panel, rain cover, hydration bladder sleeve, trekking pole attachment, and multiple compression straps. Weighs 3.2 lbs."},
    {"id": "P015", "name": "Smart Scale", "description": "WiFi-connected smart body composition scale measuring weight, BMI, body fat, muscle mass, and bone density. Syncs with Apple Health, Google Fit, and Fitbit."},
    {"id": "P016", "name": "Desk Lamp", "description": "LED desk lamp with wireless charging base, 5 color temperatures, 7 brightness levels, 60-minute auto-off timer, and USB charging port. Eye-care certified."},
    {"id": "P017", "name": "Yoga Block Set", "description": "High-density EVA foam yoga blocks, set of 2, with beveled edges and non-slip surface. Dimensions 9x6x4 inches. Supports up to 250 lbs. Includes strap."},
    {"id": "P018", "name": "Dash Cam", "description": "Dual-channel dash cam with 4K front and 1080p rear, GPS logging, night vision, parking mode with motion detection, and 64GB built-in storage. Loop recording."},
    {"id": "P019", "name": "Travel Pillow", "description": "Memory foam travel pillow with 360-degree neck support, removable washable cover, snap closure, and compact carrying bag. Weighs 10 oz."},
    {"id": "P020", "name": "Sous Vide Cooker", "description": "Precision sous vide immersion circulator with WiFi control, 1100W heating element, 0.1°F accuracy, IPX7 waterproof, and compatible with up to 8-gallon containers."}
]

print(f"Total products: {len(more_products)}")

Total products: 20


In [32]:
# Build the batch JSONL with our long system prompt
combo_requests = []
for product in more_products:
    request = {
        "custom_id": f"combo-{product['id']}",
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": MODEL,
            "instructions": long_system_prompt,
            "input": f"Product: {product['name']}\nDescription: {product['description']}"
        }
    }
    combo_requests.append(request)

combo_file_path = "batch_combo_input.jsonl"
with open(combo_file_path, "w") as f:
    for req in combo_requests:
        f.write(json.dumps(req) + "\n")

print(f"Created {combo_file_path} with {len(combo_requests)} requests")

Created batch_combo_input.jsonl with 20 requests


In [33]:
# Upload and submit
combo_file = client.files.create(file=open(combo_file_path, "rb"), purpose="batch")
combo_job = client.batches.create(
    input_file_id=combo_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
    metadata={"description": "Caching + Batch combo — 20 products"}
)
print(f"Batch ID: {combo_job.id} | Status: {combo_job.status}")

Batch ID: batch_69b24003f4c48190b06fb6507794201f | Status: validating


In [34]:
# Poll until complete
poll_start = time.time()
while True:
    job = client.batches.retrieve(combo_job.id)
    elapsed = time.time() - poll_start
    completed = job.request_counts.completed if job.request_counts else 0
    total = job.request_counts.total if job.request_counts else 0
    print(f"[{elapsed:6.1f}s] Status: {job.status:15s} | Progress: {completed}/{total}")
    if job.status in ["completed", "failed", "expired", "cancelled"]:
        break
    time.sleep(10)

print(f"\n✅ Finished in {elapsed:.1f}s")

[   0.7s] Status: validating      | Progress: 0/0
[  12.3s] Status: validating      | Progress: 0/0
[  24.1s] Status: validating      | Progress: 0/0
[  35.9s] Status: validating      | Progress: 0/0
[  47.6s] Status: validating      | Progress: 0/0
[  59.3s] Status: validating      | Progress: 0/0
[  70.8s] Status: in_progress     | Progress: 0/20
[  82.4s] Status: in_progress     | Progress: 0/20
[  94.1s] Status: in_progress     | Progress: 0/20
[ 105.7s] Status: in_progress     | Progress: 0/20
[ 117.3s] Status: in_progress     | Progress: 0/20
[ 128.9s] Status: in_progress     | Progress: 0/20
[ 140.5s] Status: in_progress     | Progress: 0/20
[ 152.3s] Status: in_progress     | Progress: 0/20
[ 163.9s] Status: in_progress     | Progress: 0/20
[ 175.7s] Status: in_progress     | Progress: 0/20
[ 187.5s] Status: in_progress     | Progress: 0/20
[ 199.0s] Status: in_progress     | Progress: 0/20
[ 210.6s] Status: in_progress     | Progress: 0/20
[ 222.5s] Status: in_progress     | P

In [35]:
# Download and parse combo results
if job.status == "completed" and job.output_file_id:
    combo_output = client.files.content(job.output_file_id).text
    combo_results = [json.loads(line) for line in combo_output.strip().split("\n")]
    
    combo_parsed = []
    for result in combo_results:
        usage = result["response"]["body"].get("usage", {})
        combo_parsed.append({
            "custom_id": result["custom_id"],
            "input_tokens": usage.get("input_tokens", 0),
            "cached_tokens": usage.get("input_tokens_details", {}).get("cached_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0)
        })
    
    combo_df = pd.DataFrame(combo_parsed)
    display(combo_df)
    
    total_input = combo_df["input_tokens"].sum()
    total_cached = combo_df["cached_tokens"].sum()
    print(f"\nTotal input tokens: {total_input:,}")
    print(f"Cached tokens:      {total_cached:,} ({total_cached/total_input*100:.1f}%)")

,custom_id,input_tokens,cached_tokens,output_tokens
0,combo-P001,1249,1152,2269
1,combo-P002,1252,1152,6351
2,combo-P003,1250,1152,3195
3,combo-P004,1250,1152,3810
4,combo-P005,1259,0,4398
5,combo-P006,1254,0,3518
6,combo-P007,1253,1152,3419
7,combo-P008,1256,1152,5218
8,combo-P009,1259,0,2919
9,combo-P010,1252,1152,3612



Total input tokens: 25,072
Cached tokens:      13,824 (55.1%)


### Cost Comparison: All Three Approaches

In [36]:
def estimate_cost_at_scale(n_products, input_per_req, output_per_req, cached_pct, batch_discount=False):
    """Estimate cost for a given approach at scale."""
    total_input = n_products * input_per_req
    cached = int(total_input * cached_pct)
    uncached = total_input - cached
    total_output = n_products * output_per_req
    
    # Apply batch discount (50% off everything)
    multiplier = 0.5 if batch_discount else 1.0
    
    input_cost = ((uncached / 1e6) * INPUT_COST_PER_M + (cached / 1e6) * CACHED_COST_PER_M) * multiplier
    output_cost = (total_output / 1e6) * OUTPUT_COST_PER_M * multiplier
    
    return input_cost + output_cost

# Estimate based on our actual token counts
avg_input = 1200   # approximate: ~1100 system + ~100 user
avg_output = 250    # typical for our structured output

scenarios = {
    "Naive (short prompt, sync)": estimate_cost_at_scale(10000, 150, avg_output, 0.0, False),
    "Long prompt + caching (sync)": estimate_cost_at_scale(10000, avg_input, avg_output, 0.80, False),
    "Long prompt + Batch API": estimate_cost_at_scale(10000, avg_input, avg_output, 0.80, True),
}

print("=== Estimated Cost for 10,000 Products ===")
print(f"{'Approach':<40} {'Cost':>10}")
print("-" * 52)
for approach, cost in scenarios.items():
    print(f"{approach:<40} ${cost:>8.2f}")

baseline = scenarios["Naive (short prompt, sync)"]
best = scenarios["Long prompt + Batch API"]
print(f"\n💰 Maximum savings: {(1 - best/baseline)*100:.0f}% (${baseline - best:.2f} saved)")

=== Estimated Cost for 10,000 Products ===
Approach                                       Cost
----------------------------------------------------
Naive (short prompt, sync)               $    1.07
Long prompt + caching (sync)             $    1.17
Long prompt + Batch API                  $    0.58

💰 Maximum savings: 46% ($0.49 saved)


---
## Section 5: Helper Utilities — Reusable Batch Pipeline (15 min)

Let's wrap everything we've learned into clean, reusable functions.

In [37]:
def create_batch_job(
    client: OpenAI,
    items: list[dict],
    system_prompt: str,
    model: str = "gpt-4o-mini",
    input_formatter=None,
    id_field: str = "id",
    description: str = "Batch job"
) -> str:
    """
    Create and submit a batch job for a list of items.
    
    Args:
        client: OpenAI client
        items: list of dicts to process
        system_prompt: the instructions (static prefix)
        model: model to use
        input_formatter: function(item) -> str that formats each item into the user input
        id_field: key in each item to use as the custom_id
        description: metadata description for the batch
    
    Returns:
        batch_id: the ID of the created batch job
    """
    if input_formatter is None:
        input_formatter = lambda item: json.dumps(item)
    
    # Build JSONL
    requests = []
    for item in items:
        requests.append({
            "custom_id": str(item.get(id_field, items.index(item))),
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": model,
                "instructions": system_prompt,
                "input": input_formatter(item)
            }
        })
    
    file_path = f"batch_input_{int(time.time())}.jsonl"
    with open(file_path, "w") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
    
    # Upload & submit
    uploaded = client.files.create(file=open(file_path, "rb"), purpose="batch")
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={"description": description}
    )
    
    print(f"Batch created: {batch.id} ({len(requests)} requests)")
    return batch.id


def wait_for_batch(client: OpenAI, batch_id: str, poll_interval: int = 10) -> dict:
    """
    Poll a batch job until completion and return parsed results.
    """
    start = time.time()
    while True:
        job = client.batches.retrieve(batch_id)
        elapsed = time.time() - start
        completed = job.request_counts.completed if job.request_counts else 0
        total = job.request_counts.total if job.request_counts else 0
        print(f"  [{elapsed:6.1f}s] {job.status:15s} ({completed}/{total})")
        
        if job.status in ["completed", "failed", "expired", "cancelled"]:
            break
        time.sleep(poll_interval)
    
    if job.status != "completed" or not job.output_file_id:
        raise RuntimeError(f"Batch failed with status: {job.status}")
    
    # Download & parse
    output_text = client.files.content(job.output_file_id).text
    results = {}
    for line in output_text.strip().split("\n"):
        r = json.loads(line)
        # Extract output text
        output = ""
        for item in r["response"]["body"].get("output", []):
            if item.get("type") == "message":
                for c in item.get("content", []):
                    if c.get("type") == "output_text":
                        output = c["text"]
        results[r["custom_id"]] = {
            "output": output,
            "usage": r["response"]["body"].get("usage", {})
        }
    
    print(f"✅ Done — {len(results)} results")
    return results

In [38]:
# Quick test of our utility
test_batch_id = create_batch_job(
    client=client,
    items=products[:3],
    system_prompt=long_system_prompt,
    model=MODEL,
    input_formatter=lambda p: f"Product: {p['name']}\nDescription: {p['description']}",
    id_field="id",
    description="Utility test — 3 products"
)

test_results = wait_for_batch(client, test_batch_id)

# Show one result
first_key = list(test_results.keys())[0]
print(f"\nResult for {first_key}:")
print(test_results[first_key]["output"][:300])

Batch created: batch_69b2518ffa208190954307cf6f4acc9e (3 requests)
  [   0.5s] validating      (0/0)
  [  12.5s] in_progress     (0/3)
  [  24.4s] in_progress     (0/3)
  [  36.4s] in_progress     (0/3)
  [  48.2s] in_progress     (0/3)
  [  60.1s] in_progress     (0/3)
  [  72.1s] in_progress     (0/3)
  [  83.9s] in_progress     (0/3)
  [  95.4s] in_progress     (0/3)
  [ 106.9s] in_progress     (0/3)
  [ 118.5s] in_progress     (0/3)
  [ 130.8s] in_progress     (0/3)
  [ 142.6s] in_progress     (0/3)
  [ 154.1s] in_progress     (0/3)
  [ 165.9s] in_progress     (0/3)
  [ 177.5s] in_progress     (0/3)
  [ 189.3s] in_progress     (0/3)
  [ 200.8s] in_progress     (0/3)
  [ 212.3s] in_progress     (0/3)
  [ 223.9s] in_progress     (0/3)
  [ 235.7s] in_progress     (0/3)
  [ 247.7s] in_progress     (0/3)
  [ 259.3s] in_progress     (0/3)
  [ 271.0s] in_progress     (0/3)
  [ 282.6s] in_progress     (0/3)
  [ 294.2s] in_progress     (0/3)
  [ 306.0s] in_progress     (0/3)
  [ 317.7s] in_

APIConnectionError: Connection error.

---
## Section 7: Best Practices & When to Use What (10 min)

### Decision Framework

```
         Need instant response?
              /           \
           YES             NO
            |               |
     Use Responses API   Use Batch API
     (synchronous)       (50% cheaper)
            |               |
   Long static prefix?    Long static prefix?
        /       \            /       \
      YES       NO         YES       NO
       |         |          |         |
   Caching    Standard   Caching    Standard
   saves $    pricing    + Batch    Batch
   + speed              = MAX $$$   saves 50%
```

### Prompt Caching Best Practices

**DO:**
- Put all static content (instructions, examples, schema) at the beginning of the prompt
- Keep the system prompt frozen — version it as configuration
- Maintain steady request flow to prevent cache eviction (5-10 min inactivity)
- Monitor `cached_tokens` in your logging pipeline

**DON'T:**
- Embed dynamic data (user name, timestamps) inside the system prompt
- Change the system prompt for A/B testing without understanding cache implications
- Expect caching on prompts under 1,024 tokens

### Batch API Best Practices

**DO:**
- Use for background jobs: catalog enrichment, document processing, evaluation runs
- Include meaningful `custom_id` values for matching results to inputs
- Add `metadata` for tracking and auditing batch jobs
- Build retry logic for expired/failed requests

**DON'T:**
- Use for user-facing features that need instant responses
- Mix models within a single batch file
- Submit batches without testing the prompt on a few synchronous calls first

### Quick Reference: Combining Both

| Scenario | Approach | Why |
|---|---|---|
| Live chatbot | Sync + caching | Need instant response; system prompt is long & static |
| Nightly catalog enrichment | Batch + caching | No urgency; max savings from both discounts |
| Quick prototype testing | Sync only | Fast iteration; caching builds up naturally |
| Evaluation / evals pipeline | Batch only | Thousands of test cases; no latency constraint |

### 🧠 Final Knowledge Check

1. **Which combination gives the maximum cost savings?**  
   a) Short prompt + synchronous API  
   b) Long prompt with caching + Batch API ✅  
   c) Short prompt + Batch API  

2. **When should you NOT use the Batch API?**  
   a) Processing 10,000 product descriptions overnight  
   b) Powering a live customer chat interface ✅  
   c) Running an LLM evaluation suite  

3. **Prompt caching is automatically invalidated when:**  
   a) You change the user message  
   b) You change any character in the cached prefix ✅  
   c) You switch from Responses API to Chat Completions API  

4. **The Batch API endpoint in the JSONL file for the Responses API is:**  
   a) `/v1/chat/completions`  
   b) `/v1/responses` ✅  
   c) `/v1/batch`

---
## Key Takeaways

1. **Prompt Caching** is automatic and free to use — just make sure your static prefix exceeds 1,024 tokens. Cached input tokens get a 50% discount and significantly lower latency.

2. **Batch API** gives you 50% off everything for workloads that don't need instant responses. Bundle requests into JSONL, submit, and poll for results.

3. **They stack** — use a long cached system prompt inside Batch API requests for maximum savings.

4. **Design for caching** — static content first, dynamic content last. Version your system prompts like config files.

5. **Use the Responses API** (`client.responses.create()`) — it's OpenAI's latest recommended API and supports both caching and batch processing.

---

## References

- [OpenAI Prompt Caching Guide](https://platform.openai.com/docs/guides/prompt-caching)
- [OpenAI Batch API Guide](https://platform.openai.com/docs/guides/batch)
- [OpenAI Responses API Reference](https://platform.openai.com/docs/api-reference/responses/create)
- [Prompt Caching Announcement](https://openai.com/index/api-prompt-caching/)
- [Batch API FAQ](https://help.openai.com/en/articles/9197833-batch-api-faq)
- [Responses API Migration Guide](https://developers.openai.com/api/docs/guides/migrate-to-responses/)